In [1]:
# CELL 1: Environment & Thread Locks
import os
import torch

os.environ["CACHE_LOOKBACK"] = "41"
os.environ["CACHE_START_DATE"] = "1998-01-01"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# Prevent sub-process thread explosion across CPU cores
torch.set_num_threads(1)

In [2]:
# CELL 3: Path Setup
import sys
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

In [3]:
# CELL 4: Imports
import torch
import numpy as np
import pandas as pd
from core.settings import TradingConfig, CacheConfig
from core.paths import OUTPUT_DIR, LOCAL_DATA_DIR
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar, get_chronological_splits
from data_pipeline.screener import UniverseScreener
from rl_discovery.oracle import RLOracle
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.adapter import RLVRGymEnv, ObservationScaler
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.trainer import RolloutBuffer, PPOTrainer
from rl_discovery.validator import AgentEvaluator

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output



In [ ]:
# CELL 5: Load Data & Memory-Optimized Environment Splits
import gc

print("Loading preprocessed data & AlphaCache...")
data = load_processed_data()
df_ohlcv = data.df_ohlcv
macro_df = data.macro_df
features_df = data.features_df
config = TradingConfig()
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()
print(f"cache_file_path:\n{cache_file_path}\n")
feature_cube = pd.read_parquet(cache_file_path)

Loading preprocessed data & AlphaCache...
cache_file_path:
C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data\alpha_cache_41d_1998.parquet



In [5]:
print(f"df_ohlcv.info():\n{df_ohlcv.info()}")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 8542534 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-09-04 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     float64
dtypes: float64(5)
memory usage: 358.8+ MB
df_ohlcv.info():
None


In [6]:
print(f"macro_df.info():\n{macro_df.info()}")

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 7466 entries, 1997-01-02 to 2026-09-04
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Mkt_Ret              7466 non-null   float64
 1   Mkt_Ret_Z            7466 non-null   float64
 2   Macro_Trend          7466 non-null   float64
 3   Macro_Trend_Z        7466 non-null   float64
 4   Yield_Curve_10Y2Y_Z  7466 non-null   float64
 5   Macro_Trend_Vel_Z    7466 non-null   float64
 6   Macro_Trend_Mom      7466 non-null   float64
 7   Macro_Vix_Z          7466 non-null   float64
 8   Macro_Vix_Ratio      7466 non-null   float32
 9   Mkt_Vol_63d_Z        7466 non-null   float64
dtypes: float32(1), float64(9)
memory usage: 612.4 KB
macro_df.info():
None


In [7]:
print(f"features_df.info():\n{features_df.info()}")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 8542534 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-09-04 00:00:00'))
Data columns (total 21 columns):
 #   Column               Dtype  
---  ------               -----  
 0   ATR                  float64
 1   ATRP                 float64
 2   TRP                  float64
 3   RSI                  float64
 4   Mom_21               float64
 5   Mom_63               float64
 6   Mom_126              float64
 7   Mom_252_21           float64
 8   Consistency          float64
 9   IR_63                float64
 10  Beta_63              float64
 11  IVol_63              float64
 12  SemiDev_63           float64
 13  Trend_R2_63          float64
 14  DD_21                float64
 15  Ret_1d               float64
 16  RollingStalePct      float64
 17  RollMedDollarVol     float64
 18  RollingSameVolCount  float64
 19  RecentStaleDays      float64
 20  IsZeroPrice          int64  
dtypes: float64(20), int64(1)
mem

In [8]:
print(f"df_close.info():\n{df_close.info()}")

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 7466 entries, 1997-01-02 to 2026-09-04
Columns: 1708 entries, A to ZWS
dtypes: float64(1708)
memory usage: 97.3 MB
df_close.info():
None


In [9]:
print(f"feature_cube.info():\n{feature_cube.info()}")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 5035126 entries, (Timestamp('1998-01-02 00:00:00'), 'AA') to (Timestamp('2026-09-04 00:00:00'), 'ZTS')
Data columns (total 12 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   41d_Momentum (12-1m)             float64
 1   41d_Sharpe (TRP)                 float64
 2   41d_Momentum (21d)               float64
 3   41d_Info Ratio (63d)             float64
 4   41d_Oversold (-RSI)              float64
 5   41d_Dip Buyer (-dd_21)           float64
 6   41d_Trend Quality (63d)          float64
 7   41d_Downside Risk (-SemiDev_63)  float64
 8   41d_Low Volatility (-ATRP)       float64
 9   41d_Momentum (63d)               float64
 10  41d_Momentum (126d)              float64
 11  41d_Residual Low-Vol (63d)       float64
dtypes: float64(12)
memory usage: 480.5+ MB
feature_cube.info():
None


In [10]:
df_ohlcv.loc["QQQ"]

,Adj Open,Adj High,Adj Low,Adj Close,Volume
Date,,,,,
1999-03-10,43.0797,43.1061,42.3687,43.0271,6209093.0
1999-03-11,43.3430,43.5932,42.3951,43.2377,11497979.0
1999-03-12,43.0797,43.1060,41.8421,42.1844,10376497.0
1999-03-15,42.5004,43.4484,42.0528,43.3957,7558431.0
1999-03-16,43.5800,43.9487,43.1061,43.7644,5821973.0
...,...,...,...,...,...
2026-08-31,715.1700,717.5800,713.1600,716.7600,32237200.0
2026-09-01,707.3900,712.3000,704.6600,707.6400,35236100.0
2026-09-02,707.1000,709.8000,705.1000,709.2400,23476200.0
